# 05 · Freeze operational policy candidates

Freezes q80/q90/q95 uncertainty/applicability cut-offs and NORMAL/DEGRADED/ABSTAIN candidate policies before policy validation.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Freeze candidate Gray-Box operational-policy inputs and thresholds using ONLY
the reserved 72-hour conformal_calibration block.

No policy-validation labels are read.
No model fitting/retraining.
No GPU use.
No deletion.
No GCS mutation.

This stage freezes:
- uncertainty/applicability signals,
- 80th/90th/95th percentile candidate cut-offs,
- a small prespecified NORMAL / DEGRADED / ABSTAIN policy family,
before policy-validation outcomes are opened.
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import numpy as np
import pandas as pd

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

CONF = Path("/home/abmoses2000/graybox_aia72_conformal_v1_20261005")
CONF_PROTOCOL = CONF / "protocol.json"
CONF_THRESHOLDS = CONF / "thresholds.csv"
CONF_CASES = CONF / "conformal_calibration_cases.csv.gz"

OUT = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

EXPECTED_ROWS = 4168
PRIMARY_ALPHA = 0.10
PERCENTILES = [0.80, 0.90, 0.95]
SEEDS = [17, 29, 43]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p = np.asarray(p, dtype=float)
    eps = 1e-12
    q = np.clip(p, eps, 1 - eps)
    return -(q * np.log(q) + (1 - q) * np.log(1 - q))

def conformal_state(p, qhat):
    """
    Binary split-conformal set for score 1-p_y:
      include class 0 iff p <= qhat
      include class 1 iff 1-p <= qhat
    Returns: empty, singleton_no_flare, singleton_flare, doubleton
    """
    include0 = p <= qhat
    include1 = (1.0 - p) <= qhat
    out = np.empty(len(p), dtype=object)
    out[(~include0) & (~include1)] = "empty"
    out[include0 & (~include1)] = "singleton_no_flare"
    out[(~include0) & include1] = "singleton_flare"
    out[include0 & include1] = "doubleton"
    return out

def quantile_higher(x, q):
    # Deterministic conservative observed cut-off.
    try:
        return float(np.quantile(x, q, method="higher"))
    except TypeError:
        return float(np.quantile(x, q, interpolation="higher"))

# ---------- Integrity ----------
for p in [MASTER, SHARP, CONF_PROTOCOL, CONF_THRESHOLDS, CONF_CASES]:
    require(p.is_file(), f"Missing required input: {p}")
require(sha256(MASTER) == MASTER_SHA, "Master SHA changed")
require(sha256(SHARP) == SHARP_SHA, "SHARP SHA changed")

protocol = json.loads(CONF_PROTOCOL.read_text())
require(protocol["status"] == "CONFORMAL_THRESHOLDS_FROZEN", "Conformal protocol not frozen")
require(protocol["primary_alpha"] == PRIMARY_ALPHA, "Primary alpha changed")
require(protocol["policy_validation_outcomes_read"] is False, "Conformal stage read policy outcomes")
require(protocol["later_outcomes_read"] is False, "Conformal stage read later outcomes")

thresholds = pd.read_csv(CONF_THRESHOLDS)
conf_cases = pd.read_csv(CONF_CASES, low_memory=False)
require(len(conf_cases) == EXPECTED_ROWS, "Conformal case count changed")
require(conf_cases["forecast_case_id"].is_unique, "Conformal case IDs not unique")

# ---------- Read only conformal_calibration rows ----------
master = pd.read_csv(MASTER, low_memory=False)
sharp = pd.read_csv(SHARP, low_memory=False)

aia = master.loc[master["role"].astype(str) == "conformal_calibration"].copy()
sh = sharp.loc[sharp["role"].astype(str) == "conformal_calibration"].copy()

require(len(aia) == EXPECTED_ROWS, "AIA conformal support changed")
require(len(sh) == EXPECTED_ROWS, "SHARP conformal support changed")

aia["forecast_case_id"] = aia["forecast_case_id"].astype(str)
sh["forecast_case_id"] = sh["forecast_case_id"].astype(str)

require(
    aia["forecast_case_id"].tolist() == sh["forecast_case_id"].tolist(),
    "SHARP/AIA conformal IDs/order mismatch"
)
require(
    aia["forecast_case_id"].tolist() == conf_cases["forecast_case_id"].astype(str).tolist(),
    "Frozen conformal-case order mismatch"
)

# ---------- Seed probabilities ----------
aia_seed_cols = [f"probability_seed_{s}" for s in SEEDS]
sharp_seed_cols = [f"probability_seed_{s}" for s in SEEDS]
for c in aia_seed_cols:
    require(c in aia.columns, f"AIA missing {c}")
for c in sharp_seed_cols:
    require(c in sh.columns, f"SHARP missing {c}")
require("probability" in aia.columns, "AIA missing mean probability")
require("probability_gru_mean" in sh.columns, "SHARP missing GRU mean probability")

aia_seed = aia[aia_seed_cols].astype(float).to_numpy()
sharp_seed = sh[sharp_seed_cols].astype(float).to_numpy()

for name, arr in [("aia_seed", aia_seed), ("sharp_seed", sharp_seed)]:
    require(np.isfinite(arr).all(), f"{name} contains non-finite values")
    require(((0 <= arr) & (arr <= 1)).all(), f"{name} outside [0,1]")

aia_mean = aia["probability"].astype(float).to_numpy()
sharp_mean = sh["probability_gru_mean"].astype(float).to_numpy()

# Verify stored means equal the corresponding 3-seed means to numerical tolerance.
require(
    np.allclose(aia_seed.mean(axis=1), aia_mean, rtol=1e-10, atol=1e-12),
    "AIA stored mean does not match 3-seed mean"
)
require(
    np.allclose(sharp_seed.mean(axis=1), sharp_mean, rtol=1e-10, atol=1e-12),
    "SHARP stored mean does not match 3-seed mean"
)

fusion_seed = 0.5 * sharp_seed + 0.5 * aia_seed
fusion_mean = fusion_seed.mean(axis=1)
require(
    np.allclose(fusion_mean, 0.5 * sharp_mean + 0.5 * aia_mean, rtol=1e-10, atol=1e-12),
    "Fusion seed mean mismatch"
)

# ---------- Signals ----------
signals = pd.DataFrame({
    "forecast_case_id": aia["forecast_case_id"].to_numpy(),
    "issue_utc": aia["issue_utc"].to_numpy() if "issue_utc" in aia.columns else pd.NA,
    "sharp_probability": sharp_mean,
    "aia_probability": aia_mean,
    "fusion_probability": fusion_mean,
    "sharp_seed_std": sharp_seed.std(axis=1, ddof=1),
    "aia_seed_std": aia_seed.std(axis=1, ddof=1),
    "fusion_seed_std": fusion_seed.std(axis=1, ddof=1),
    "sharp_aia_probability_gap": np.abs(sharp_mean - aia_mean),
    "sharp_entropy": entropy_binary(sharp_mean),
    "aia_entropy": entropy_binary(aia_mean),
    "fusion_entropy": entropy_binary(fusion_mean),
})

# Preserve AIA input quality without looking at labels.
if "input_status" in aia.columns:
    signals["aia_input_status"] = aia["input_status"].astype(str).to_numpy()
else:
    signals["aia_input_status"] = "unknown"

if "input_failure_reason" in aia.columns:
    signals["aia_input_failure_reason"] = aia["input_failure_reason"].astype(str).to_numpy()
else:
    signals["aia_input_failure_reason"] = ""

# SHARP predictions.csv.gz contains only accepted prediction rows; mark as available.
signals["sharp_input_status"] = "ok"

# ---------- Primary conformal states ----------
qhat = {}
for branch in ["sharp", "aia", "sharp_aia"]:
    hit = thresholds.loc[
        (thresholds["branch"] == branch) &
        (np.isclose(thresholds["alpha"].astype(float), PRIMARY_ALPHA))
    ]
    require(len(hit) == 1, f"Missing/duplicate primary conformal threshold for {branch}")
    qhat[branch] = float(hit.iloc[0]["qhat_nonconformity"])

signals["sharp_conformal_state"] = conformal_state(sharp_mean, qhat["sharp"])
signals["aia_conformal_state"] = conformal_state(aia_mean, qhat["aia"])
signals["fusion_conformal_state"] = conformal_state(fusion_mean, qhat["sharp_aia"])

# ---------- Freeze percentile cut-offs ----------
metric_names = [
    "sharp_seed_std",
    "aia_seed_std",
    "fusion_seed_std",
    "sharp_aia_probability_gap",
    "fusion_entropy",
]

cutoff_rows = []
for metric in metric_names:
    x = signals[metric].to_numpy(float)
    require(np.isfinite(x).all(), f"Non-finite policy signal: {metric}")
    for q in PERCENTILES:
        cutoff_rows.append({
            "metric": metric,
            "percentile": q,
            "cutoff": quantile_higher(x, q),
            "calibration_rows": len(x),
            "source_role": "conformal_calibration",
            "labels_used": False,
        })

cutoffs = pd.DataFrame(cutoff_rows)

# ---------- Freeze candidate policy family ----------
# Policy-validation may choose among these candidates later, but may not invent
# new percentile cut-offs after seeing outcomes.
candidate_policies = []

for pct in PERCENTILES:
    c = {
        row.metric: float(row.cutoff)
        for row in cutoffs.loc[np.isclose(cutoffs["percentile"], pct)].itertuples()
    }

    policy_id = f"policy_q{int(pct*100)}"
    candidate_policies.append({
        "policy_id": policy_id,
        "cutoff_percentile": pct,
        "primary_alpha": PRIMARY_ALPHA,
        "normal_rule": {
            "all": [
                "aia_input_status == 'ok'",
                "fusion_conformal_state in {'singleton_no_flare','singleton_flare'}",
                f"fusion_seed_std <= {c['fusion_seed_std']:.17g}",
                f"sharp_aia_probability_gap <= {c['sharp_aia_probability_gap']:.17g}",
                f"fusion_entropy <= {c['fusion_entropy']:.17g}",
            ]
        },
        "degraded_rule": {
            "all": [
                "NORMAL rule is false",
                "sharp_input_status == 'ok'",
                "sharp_conformal_state in {'singleton_no_flare','singleton_flare'}",
                f"sharp_seed_std <= {c['sharp_seed_std']:.17g}",
            ],
            "forecast_source": "SHARP fallback",
        },
        "abstain_rule": "Neither NORMAL nor DEGRADED rule is satisfied",
        "normal_forecast_source": "fixed 0.5/0.5 SHARP+AIA fusion",
        "notes": [
            "Policy uses no labels at freeze time.",
            "AIA-quality failure can force fallback to SHARP if SHARP passes its frozen trust checks.",
            "Empty or doubleton fusion conformal state cannot be NORMAL.",
            "Later policy-validation may select among q80/q90/q95 only; it may not tune new cut-offs on the same outcomes.",
        ],
    })

baseline = {
    "policy_id": "baseline_always_issue_fusion_when_available",
    "rule": "Issue fixed 0.5/0.5 fusion whenever both modalities are available; no uncertainty gating.",
    "purpose": "Operational comparator for policy validation.",
}

# ---------- Descriptive label-free state counts on calibration block ----------
state_counts = {}
for branch in ["sharp","aia","fusion"]:
    col = f"{branch}_conformal_state"
    state_counts[branch] = {
        str(k): int(v)
        for k, v in signals[col].value_counts(dropna=False).to_dict().items()
    }

signals_path = OUT / "calibration_policy_signals.csv.gz"
cutoffs_path = OUT / "candidate_cutoffs.csv"
spec_path = OUT / "policy_family.json"
protocol_path = OUT / "protocol.json"

signals.to_csv(
    signals_path, index=False,
    compression={"method": "gzip", "mtime": 0}
)
cutoffs.to_csv(cutoffs_path, index=False)

policy_family = {
    "status": "POLICY_CANDIDATE_FAMILY_FROZEN",
    "scientific_acceptance": False,
    "baseline": baseline,
    "candidates": candidate_policies,
}
spec_path.write_text(json.dumps(policy_family, indent=2) + "\n")

freeze_protocol = {
    "status": "POLICY_INPUTS_AND_CANDIDATES_FROZEN",
    "scientific_acceptance": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "horizon_hours": 72,
    "source_role": "conformal_calibration",
    "rows": EXPECTED_ROWS,
    "labels_read_for_threshold_freeze": False,
    "policy_validation_outcomes_read": False,
    "later_outcomes_read": False,
    "primary_conformal_alpha": PRIMARY_ALPHA,
    "candidate_percentiles": PERCENTILES,
    "signals": metric_names + [
        "sharp_conformal_state",
        "aia_conformal_state",
        "fusion_conformal_state",
        "aia_input_status",
        "sharp_input_status",
    ],
    "candidate_policy_ids": [x["policy_id"] for x in candidate_policies],
    "baseline_policy_id": baseline["policy_id"],
    "selection_rule_for_next_stage": (
        "Evaluate only the frozen baseline and q80/q90/q95 candidates on policy_validation. "
        "Select under a prespecified operational objective that jointly reports skill, "
        "flare-event retention/missed flares, false alarms, coverage/availability, degraded-use rate, "
        "and abstention rate; do not select on accepted-only skill alone."
    ),
    "source_sha256": {
        "master_predictions": MASTER_SHA,
        "sharp_predictions": SHARP_SHA,
        "conformal_protocol": sha256(CONF_PROTOCOL),
        "conformal_thresholds": sha256(CONF_THRESHOLDS),
        "conformal_cases": sha256(CONF_CASES),
    },
    "conformal_state_counts_on_calibration": state_counts,
    "output_sha256": {
        signals_path.name: sha256(signals_path),
        cutoffs_path.name: sha256(cutoffs_path),
        spec_path.name: sha256(spec_path),
    },
}
protocol_path.write_text(json.dumps(freeze_protocol, indent=2) + "\n")

print("===== POLICY FREEZE COMPLETE =====")
print("rows:", EXPECTED_ROWS)
print("labels used for threshold freeze: False")
print("policy-validation outcomes read: False")

print("\nCandidate cut-offs:")
print(cutoffs.to_string(index=False))

print("\nPrimary-alpha conformal state counts:")
print(json.dumps(state_counts, indent=2))

print("\nFrozen candidate policies:")
for p in candidate_policies:
    print("\n", p["policy_id"])
    print("  NORMAL:", p["normal_rule"])
    print("  DEGRADED:", p["degraded_rule"])
    print("  ABSTAIN:", p["abstain_rule"])

print("\nProtocol:")
print(json.dumps(freeze_protocol, indent=2))

print("\nSaved:", protocol_path)
print("Saved:", spec_path)
print("Saved:", cutoffs_path)
print("Saved:", signals_path)
